# Decision trees and random forests

Data Scientist track, lesson DS7. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** read the questions of a decision tree, limit its depth and leaf size with cross-validation, average many trees in a random forest, and read feature importances with caution.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the app-events data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it. The data is synthetic: a script made it from stated rules.

## Setup

In [ ]:
import math
import numbers
import os
import subprocess
import sys
import tempfile
import urllib.request

import numpy as np
import pandas as pd

try:
    import duckdb
except ImportError:  # Colab usually has DuckDB; install it when it does not
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
    import duckdb
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.tree import DecisionTreeClassifier, export_text

SEED = 2026       # the seed of the whole Data Scientist track

EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3']


# Run this cell as it is. It also holds the answer checker used by the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.

RESULTS = {}


def same(got, expected, tol=1e-9):
    """True when two values agree: floats within `tol`, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k], tol) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e, tol) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=tol)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None, tol=1e-9):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected, tol):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps.items() if isinstance(traps, dict) else (traps or [])):
        if same(got, wrong, tol):
            print("Hint:", message)
    return False

## Data

In [ ]:
# The app-events tables (version 1), read from the release tag v0.2.0, or from the repository's own copy of the same files when the network is not available.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.2.0/datasets/app-events/v1/"
LOCAL = "../../datasets/app-events/v1/"      # the same files, when this notebook runs inside a clone of the repository


def fetch(name):
    """The path of one CSV file: downloaded from the tag, or the local copy."""
    try:
        path = os.path.join(tempfile.mkdtemp(), name + ".csv")
        urllib.request.urlretrieve(BASE + name + ".csv", path)
        return path
    except Exception:
        return LOCAL + name + ".csv"

con = duckdb.connect()                       # a new, empty database every time this cell runs
con.execute(f"CREATE TABLE users AS FROM read_csv('{fetch('users')}', types={{'signup_date': 'DATE'}})")
con.execute(f"CREATE TABLE events AS FROM read_csv('{fetch('events')}', types={{'event_ts': 'TIMESTAMP', 'received_at': 'TIMESTAMP'}})")

# Known answer from the dataset README: 21,841 people and 177,379 events.
assert con.execute("SELECT (SELECT COUNT(*) FROM users), (SELECT COUNT(*) FROM events)").fetchall()[0] == (21841, 177379)

# The modeling table of the Data Scientist track: one row per person, built by the query of lesson DS3, "Building a modeling table in SQL".
# Run this cell as it is; DS3 explains it line by line.
MODELING_SQL = """WITH people AS (            -- one row per person: the spine of the table
  SELECT u.user_id, u.signup_date, u.channel, u.region, s.platform,
         s.event_ts AS signup_ts,
         s.event_ts + INTERVAL 7 DAY AS cutoff,       -- the prediction moment
         s.event_ts + INTERVAL 30 DAY AS horizon      -- the end of the label window
  FROM users AS u
  JOIN events AS s ON s.user_id = u.user_id AND s.event_name = 'signup_complete'
  WHERE u.signup_date BETWEEN DATE '2026-01-01' AND DATE '2026-05-31'
),
clean AS (                  -- retry copies removed: keep the first copy of each event
  SELECT *
  FROM events
  QUALIFY ROW_NUMBER() OVER (PARTITION BY user_id, event_ts, event_name, platform ORDER BY received_at, event_id) = 1
),
known AS (                  -- what the app knew at the cutoff: it had happened and it had arrived
  SELECT p.user_id, c.event_name, c.event_ts, p.cutoff
  FROM people AS p
  JOIN clean AS c ON c.user_id = p.user_id
  WHERE c.event_ts <= p.cutoff AND c.received_at <= p.cutoff
),
features AS (
  SELECT user_id,
         COUNT(*) FILTER (WHERE event_name = 'session_start') AS sessions_7d,
         COUNT(*) FILTER (WHERE event_name = 'add_expense') AS expenses_7d,
         COUNT(*) FILTER (WHERE event_name IN ('session_start', 'add_expense') AND event_ts > cutoff - INTERVAL 3 DAY) AS recent_3d,
         COUNT(DISTINCT CAST(event_ts AS DATE)) FILTER (WHERE event_name IN ('session_start', 'add_expense')) AS active_days_7d,
         CAST(COUNT(*) FILTER (WHERE event_name = 'onboarding_complete') > 0 AS INTEGER) AS onboarded,
         CAST(COUNT(*) FILTER (WHERE event_name IN ('view_paywall', 'paywall_viewed')) > 0 AS INTEGER) AS saw_paywall,
         CAST(COUNT(*) FILTER (WHERE event_name = 'purchase') > 0 AS INTEGER) AS bought_plus
  FROM known
  GROUP BY user_id
),
label AS (                  -- from the other side of the cutoff: no visit and no expense in days 8 to 30
  SELECT p.user_id,
         CAST(NOT EXISTS (SELECT 1 FROM events AS e
                          WHERE e.user_id = p.user_id AND e.event_name IN ('session_start', 'add_expense')
                            AND e.event_ts > p.cutoff AND e.event_ts <= p.horizon) AS INTEGER) AS churned
  FROM people AS p
)
SELECT p.user_id, p.signup_date, p.channel, p.platform, p.region,
       f.sessions_7d, f.expenses_7d, f.recent_3d, f.active_days_7d, f.onboarded, f.saw_paywall, f.bought_plus,
       l.churned
FROM people AS p
JOIN features AS f ON f.user_id = p.user_id
JOIN label AS l ON l.user_id = p.user_id
ORDER BY p.user_id"""
table = con.sql(MODELING_SQL).df()
print(f"Data loaded: {len(table):,} people, {int(table['churned'].sum()):,} of them churned.")

## The split, the tree and the forest

The columns, the split of lessons DS2 and DS4 (the test part stays untouched until the last unit), the five folds of lesson DS5, the day-7 model, the tree of lesson DS5 with a second knob (`leaf`), and a random forest. A tree needs no scaling, so the number columns go in as they are. Run this cell as it is.

In [ ]:
STATIC = ["channel", "platform", "region"]                    # known at signup
WEEK1 = ["sessions_7d", "expenses_7d", "recent_3d", "active_days_7d", "onboarded", "saw_paywall", "bought_plus"]    # known at day 7
X, y = table[STATIC + WEEK1], table["churned"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)       # 13,362 and 4,454 people
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=SEED, stratify=y_train)   # 10,021 and 3,341, cut from the training part
folds = StratifiedKFold(5, shuffle=True, random_state=SEED)       # the five folds of lesson DS5


def logit():
    """The day-7 model of lesson DS2: one-hot text columns, scaled number columns, logistic regression."""
    return make_pipeline(ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), STATIC), ("num", StandardScaler(), WEEK1)]), LogisticRegression(max_iter=1000))


def auc_gain(p_new, p_base, y_true, resamples=2000):
    """ROC AUC of one model minus another on the same people, with a 95 percent bootstrap interval; both models are scored on the same resamples."""
    rng = np.random.default_rng(SEED)
    y_true, a, b = np.asarray(y_true), np.asarray(p_new), np.asarray(p_base)
    gains = []
    for _ in range(resamples):
        i = rng.integers(0, len(y_true), len(y_true))
        gains.append(roc_auc_score(y_true[i], a[i]) - roc_auc_score(y_true[i], b[i]))
    low, high = np.percentile(gains, [2.5, 97.5])
    return [round(float(roc_auc_score(y_true, a) - roc_auc_score(y_true, b)), 3) + 0.0, round(float(low), 3) + 0.0, round(float(high), 3) + 0.0]


def prep(num=WEEK1):
    """One-hot text columns and the number columns as they are: a tree does not care about units."""
    return ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), STATIC), ("num", "passthrough", num)], verbose_feature_names_out=False, sparse_threshold=0)


def tree(depth=None, leaf=1):
    """A decision tree: `depth` is the most questions in a row, `leaf` the fewest people allowed in a leaf."""
    return make_pipeline(prep(), DecisionTreeClassifier(max_depth=depth, min_samples_leaf=leaf, random_state=SEED))


def forest(trees=200, leaf=100, **settings):
    """A random forest of `trees` trees, each leaf with at least `leaf` people."""
    return make_pipeline(prep(), RandomForestClassifier(n_estimators=trees, min_samples_leaf=leaf, n_jobs=-1, random_state=SEED, **settings))


print(len(X_fit), len(X_val), len(X_test))

## Unit 1: How a tree picks its questions

**Predict first.** The fitting people start at an impurity of 0.496. Which single question will lower it most: one about `channel_paid_social`, `expenses_7d` or `active_days_7d`?

In [ ]:
names = list(prep().fit(X_fit).get_feature_names_out())
Xf = pd.DataFrame(prep().fit_transform(X_fit), columns=names)         # the fitting people as 0/1 and number columns
p = y_fit.mean()
print("impurity of all the fitting people:", round(float(2 * p * (1 - p)), 3))
rows = []
for col in names:
    stump = DecisionTreeClassifier(max_depth=1, random_state=SEED).fit(Xf[[col]], y_fit)       # one question about one column
    t = stump.tree_
    after = (t.n_node_samples[1] * t.impurity[1] + t.n_node_samples[2] * t.impurity[2]) / t.n_node_samples[0]
    rows.append((col, round(float(t.threshold[0]), 1), round(float(after), 3)))
print(pd.DataFrame(rows, columns=["column", "split at", "impurity after"]).sort_values("impurity after").head(4).to_string(index=False))
two = DecisionTreeClassifier(max_depth=2, min_samples_leaf=50, random_state=SEED).fit(Xf, y_fit)
print(export_text(two, feature_names=names))

## Unit 2: Leaf size is the stronger brake

**Predict first.** A tree of depth 10 can have leaves of one person or leaves of at least 400 people. Will the two score about the same on the folds?

In [ ]:
grid = {"decisiontreeclassifier__max_depth": [2, 3, 4, 5, 6, 8, 10], "decisiontreeclassifier__min_samples_leaf": [1, 10, 50, 100, 200, 400]}
tree_search = GridSearchCV(tree(), grid, cv=folds, scoring="roc_auc").fit(X_train, y_train)
cv_table = pd.DataFrame(tree_search.cv_results_)
cv_table = cv_table.pivot(index="param_decisiontreeclassifier__max_depth", columns="param_decisiontreeclassifier__min_samples_leaf", values="mean_test_score")
cv_table.index.name, cv_table.columns.name = "depth", "leaf"
print(cv_table.loc[[3, 6, 10], [1, 50, 400]].round(3).to_string())
print(tree_search.best_params_, round(float(tree_search.best_score_), 3), round(float(cross_val_score(logit(), X_train, y_train, cv=folds, scoring="roc_auc").mean()), 3))

## Unit 3: Many trees, averaged

**Predict first.** How many trees until the validation AUC stops improving: about 5, about 25 or about 300? And will a forest with leaves of one person score the same as one with leaves of 100?

In [ ]:
forest_curve = []
for trees in [1, 5, 25, 100, 300]:
    m = forest(trees).fit(X_fit, y_fit)
    forest_curve.append((trees, round(float(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])), 3)))
print(forest_curve)
leaf_rows = []
for leaf in [1, 20, 100]:
    s = cross_val_score(forest(200, leaf), X_train, y_train, cv=folds, scoring="roc_auc")
    leaf_rows.append((leaf, round(float(s.mean()), 3), round(float(s.min()), 3), round(float(s.max()), 3)))
    print(*leaf_rows[-1])

## Unit 4: Importances with caution

**Predict first.** In a forest with leaves of one person (the setting scikit-learn starts from), where will the built-in importance rank the random column among the 20 columns: last, in the middle, or first? And what will shuffling it do to the score of held-out people?

In [ ]:
Xn = X_fit.assign(noise=np.random.default_rng(SEED).normal(size=len(X_fit)))                # a column of random numbers: nothing in it predicts churn
Xnv = X_val.assign(noise=np.random.default_rng(SEED + 1).normal(size=len(X_val)))
imp_rank, built_in_by = {}, {}
for leaf in (1, 100):
    m = make_pipeline(prep(WEEK1 + ["noise"]), RandomForestClassifier(n_estimators=200, min_samples_leaf=leaf, n_jobs=-1, random_state=SEED)).fit(Xn, y_fit)
    built_in = pd.Series(m[-1].feature_importances_, index=m[0].get_feature_names_out()).sort_values(ascending=False)       # the importance that comes with the fitted forest
    built_in_by[leaf] = built_in
    imp_rank[leaf] = (list(built_in.index).index("noise") + 1, round(float(built_in["noise"]), 3))
    print(f"leaf {leaf}: random column ranks {list(built_in.index).index('noise') + 1} of {len(built_in)} with share {built_in['noise']:.3f}; first is {built_in.index[0]} with {built_in.iloc[0]:.3f}")
held = permutation_importance(m, Xnv, y_val, scoring="roc_auc", n_repeats=10, random_state=SEED, n_jobs=-1)      # m is the forest with leaves of 100: shuffle one column of the validation people at a time
perm = pd.Series(held.importances_mean, index=Xnv.columns).sort_values(ascending=False)
print(perm.round(4).to_string())

## Unit 5: The verdict on the test part

**Predict first.** Will the forest beat the day-7 model by more than 0.005 AUC?

In [ ]:
p_day7 = logit().fit(X_train, y_train).predict_proba(X_test)[:, 1]              # the test part is used once, here
p_tree = tree_search.best_estimator_.predict_proba(X_test)[:, 1]
p_forest = forest(300, 100).fit(X_train, y_train).predict_proba(X_test)[:, 1]
print("day-7 model", round(float(roc_auc_score(y_test, p_day7)), 3), "tuned tree", round(float(roc_auc_score(y_test, p_tree)), 3), "forest", round(float(roc_auc_score(y_test, p_forest)), 3))
print("tree", auc_gain(p_tree, p_day7, y_test), "forest", auc_gain(p_forest, p_day7, y_test))

## Finish it

### Exercise 1: Finish it

**Task:** **How many columns may each split see?** Score a forest of 200 trees with leaves of at least 100 people on the five folds of the training part for each value of `max_features` in the list (given), with ROC AUC, and put the three mean scores in `answer_1`, in the order of the list, each rounded to 3 decimals.

**Expected:** `answer_1` is a list of three numbers.

In [ ]:
scores_mf = []
for mf in [1, 4, 19]:                       # given: one column, four columns, all 19 columns considered at each split
    # score forest(200, 100, max_features=mf) on the five folds of the training part with ROC AUC and keep the mean in scores_mf
    pass
answer_1 = None  # replace None with the three mean AUCs in the order of the list, each rounded to 3 decimals

In [ ]:
_ = check("Exercise 1", "answer_1", [0.687, 0.691, 0.69], traps=[([0.691, 0.691, 0.691], "That is the forest with the default number of columns for every value. Pass max_features=mf.")], tol=5e-4)

<details>
<summary>Hint</summary>

`forest(200, 100, max_features=mf)` builds the forest; `cross_val_score(..., X_train, y_train, cv=folds, scoring="roc_auc")` scores it on the five folds.

</details>

<details>
<summary>Half-solution</summary>

```python
s = cross_val_score(forest(200, 100, max_features=...), X_train, y_train, cv=folds, scoring="roc_auc")
scores_mf.append(round(float(s.mean()), 3))
```

</details>

<details>
<summary>Solution</summary>

```python
scores_mf = []
for mf in [1, 4, 19]:
    s = cross_val_score(forest(200, 100, max_features=mf), X_train, y_train, cv=folds, scoring="roc_auc")
    scores_mf.append(round(float(s.mean()), 3))
answer_1 = scores_mf
```

The scores are 0.687, 0.691, 0.690 for one, four and all 19 columns. They differ by 0.004, less than the range of the folds: on this table the forest does not care how many columns each split sees.

</details>

## On your own

### Exercise 2: On your own

**Task:** **Read a leaf.** Before you start, say in one sentence what the result looks like: a count of some hundreds of people and a churn share above 0.8. Use `Xf` (the fitting people as numbers, from unit 1). Fit a tree with `max_depth=3` and `min_samples_leaf=50` on `Xf` and `y_fit`, find the leaf where the share of churners is highest, and put the list [the number of fitting people in that leaf, their churn share rounded to 3 decimals] in `answer_2`. Write the code in the empty cell of the notebook.

**Expected:** `answer_2` is a list of two numbers.

In [ ]:
_ = check("Exercise 2", "answer_2", [877, 0.835], traps=[([3535, 0.496], "That is the largest leaf, not the leaf with the highest churn share.")], tol=5e-4)

<details>
<summary>Hint</summary>

`tree.apply(Xf)` gives the number of the leaf that each person ends in. Group the outcome by that number and take the size and the mean of each group.

</details>

<details>
<summary>Half-solution</summary>

Expected output in words: *"One leaf of a few hundred people, a little over four in five of them churners."*

```python
leaf_of = small_tree.apply(Xf)
leaf_table = pd.DataFrame({"leaf": leaf_of, "churned": y_fit.to_numpy()}).groupby("leaf")["churned"].agg(["size", "mean"])
```

</details>

<details>
<summary>Solution</summary>

```python
small_tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, random_state=SEED).fit(Xf, y_fit)
leaf_of = small_tree.apply(Xf)                                       # the leaf each fitting person ends in
leaf_table = pd.DataFrame({"leaf": leaf_of, "churned": y_fit.to_numpy()}).groupby("leaf")["churned"].agg(["size", "mean"])
top = leaf_table["mean"].idxmax()
answer_2 = [int(leaf_table.loc[top, "size"]), round(float(leaf_table.loc[top, "mean"]), 3)]
```

The leaf holds 877 fitting people and 0.835 of them churned. Printed as questions, it is the people from paid social who added no expense and did not see the paywall. The tree has 8 leaves of at least 50 people; the leaf with the highest share is not the largest one, which holds 3535 people.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** **Spot the flaw.** A colleague shuffles the four activity columns of the validation people one at a time, as in unit 4, takes the biggest drop in AUC (the code gives it), and writes that the first week's activity cannot matter by more than that. Change the last line: shuffle the four columns together, with one new order for all four, ten times, and put the mean drop in AUC in `answer_3`, rounded to 3 decimals.

**Expected:** `answer_3` is a number between 0 and 0.2.

In [ ]:
ACTIVITY = ["sessions_7d", "expenses_7d", "recent_3d", "active_days_7d"]       # given: the four columns that count the first week's activity
rf = forest(200, 100).fit(X_fit, y_fit)
base_auc = roc_auc_score(y_val, rf.predict_proba(X_val)[:, 1])
rng = np.random.default_rng(5)
single = {}
for col in ACTIVITY:                          # shuffle one column of the validation people at a time, ten times
    drops = []
    for _ in range(10):
        shuffled = X_val.copy()
        shuffled[col] = rng.permutation(shuffled[col].to_numpy())
        drops.append(base_auc - roc_auc_score(y_val, rf.predict_proba(shuffled)[:, 1]))
    single[col] = float(np.mean(drops))
answer_3 = round(max(single.values()), 3)     # colleague: "no activity column costs more than this, so together they cannot matter more"
# change the last line: shuffle the four columns together (the same new order for all four, ten times) and give the mean drop in AUC, rounded to 3 decimals

In [ ]:
_ = check("Exercise 3", "answer_3", 0.077, traps={0.02: "That is the biggest single drop. Shuffle the four columns together, with the same new order for all four."}, tol=5e-4)

<details>
<summary>Hint</summary>

When two columns say nearly the same thing, the model still reads it from the other when one is shuffled. What happens if the whole group is shuffled?

</details>

<details>
<summary>Half-solution</summary>

```python
order = rng.permutation(len(X_val))
shuffled[ACTIVITY] = X_val[ACTIVITY].to_numpy()[order]
```

</details>

<details>
<summary>Solution</summary>

```python
drops = []
for _ in range(10):
    order = rng.permutation(len(X_val))
    shuffled = X_val.copy()
    shuffled[ACTIVITY] = X_val[ACTIVITY].to_numpy()[order]            # the four columns move together
    drops.append(base_auc - roc_auc_score(y_val, rf.predict_proba(shuffled)[:, 1]))
answer_3 = round(float(np.mean(drops)), 3)
```

The biggest single drop is 0.020 and the four columns shuffled together cost 0.077, about 4 times as much. The four columns count the same week of activity in different ways, so when one is shuffled the forest reads the rest from the other three. The scikit-learn guide says this about correlated features: the model "still has access to the latter through its correlated feature". Shuffle a group of correlated columns together to see what the group is worth.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")